In [ ]:
import os
import httpx
from dotenv import load_dotenv
load_dotenv(os.path.join("..", ".env"))

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import InjectedToolArg, InjectedToolCallId, tool
from langgraph.prebuilt import InjectedState
from langgraph.types import Command
from markdownify import markdownify
from pydantic import BaseModel, Field
from tavily import TavilyClient
from typing_extensions import Annotated, Literal

import os
from datetime import datetime
import uuid, base64

In [ ]:
summarization_model = init_chat_model(model="ollama:llama3.2:3b", temperature=0.2)
tavily_client = TavilyClient(api_key="" + os.getenv("TAVILY_API_KEY"))

class Summary(BaseModel):
    """Schema for webpage content summarization."""
    filename: str = Field(description="Name of the file to store.")
    summary: str = Field(description="Key learnings from the webpage.")

def get_today_str() -> str:
    """Get current date in a human-readable format."""
    return datetime.now().strftime("%a %b %-d, %Y")

def run_tavily_search(
    search_query: str, 
    max_results: int = 1, 
    topic: Literal["general", "news", "finance"] = "general", 
    include_raw_content: bool = True, 
) -> dict:
    
    result = tavily_client.search(
        query=search_query,
        search_depth="advanced",
        max_results=max_results,
        include_domains=[
            "https://www.avma.org/",
            "https://www.merckvetmanual.com/",
            "https://www.vet.cornell.edu/",
            "https://www.vetmed.ucdavis.edu/",
            "https://vcahospitals.com/know-your-pet",
            "https://www.petmd.com/",
            "https://www.aspca.org/pet-care",
            "https://www.petpoisonhelpline.com/",
            "https://www.wormsandgermsblog.com/",
            "https://www.vin.com/"
            ],
        exclude_domains=["reddit.com", "quora.com","wikipedia.org"],
        include_raw_content=True
    )

    return result

from tools_and_prompts.prompts import SUMMARIZE_WEB_SEARCH
def summarize_webpage_content(webpage_content: str) -> Summary:
    try:
        # Set up structured output model for summarization
        structured_model = summarization_model.with_structured_output(Summary)
        
        # Generate summary
        summary_and_filename = structured_model.invoke([
            HumanMessage(content=SUMMARIZE_WEB_SEARCH.format(
                webpage_content=webpage_content, 
                date=get_today_str()
            ))
        ])
        
        return summary_and_filename
        
    except Exception:
        # Return a basic summary object on failure
        return Summary(
            filename="search_result.md",
            summary=webpage_content[:1000] + "..." if len(webpage_content) > 1000 else webpage_content
        )


def process_search_results(results: dict) -> list[dict]:
    processed_results = []
    HTTPX_CLIENT = httpx.Client()
    
    for result in results.get('results', []):
        
        url = result['url']
        response = HTTPX_CLIENT.get(url)

        if response.status_code == 200:
            # Convert HTML to markdown
            raw_content = markdownify(response.text)
            summary_obj = summarize_webpage_content(raw_content)
        else:
            # Use Tavily's generated summary
            raw_content = result.get('raw_content', '')
            summary_obj = Summary(
                filename="URL_error.md",
                summary=result.get('content', 'Error reading URL; try another search.')
            )
        
        # uniquify file names
        uid = base64.urlsafe_b64encode(uuid.uuid4().bytes).rstrip(b"=").decode("ascii")[:8]
        name, ext = os.path.splitext(summary_obj.filename)
        summary_obj.filename = f"{name}_{uid}{ext}"

        processed_results.append({
            'url': result['url'],
            'title': result['title'],
            'summary': summary_obj.summary,
            'filename': summary_obj.filename,
            'raw_content': raw_content,
        })
    
    return processed_results

In [ ]:

from tools_and_prompts.state import DeepAgentState
@tool(parse_docstring=True)
def tavily_search(
    query: str,
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
    max_results: Annotated[int, InjectedToolArg] = 1,
    topic: Annotated[Literal["general", "news", "finance"], InjectedToolArg] = "general",
) -> Command:
    """Search web and save detailed results to files while returning minimal context.

    Performs web search and saves full content to files for context offloading.
    Returns only essential information to help the agent decide on next steps.

    Args:
        query: Search query to execute
        state: Injected agent state for file storage
        tool_call_id: Injected tool call identifier
        max_results: Maximum number of results to return (default: 1)
        topic: Topic filter - 'general', 'news', or 'finance' (default: 'general')

    Returns:
        Command that saves full results to files and provides minimal summary
    """
    
    search_results = run_tavily_search(
        query,
        max_results=max_results,
        topic=topic,
        include_raw_content=True,
    ) 

    
    processed_results = process_search_results(search_results)
    
    # Save each result to a file and prepare summary
    files = state.get("files", {})
    saved_files = []
    summaries = []
    
    for i, result in enumerate(processed_results):
        # Use the AI-generated filename from summarization
        filename = result['filename']
        
        # Create file content with full details
        file_content = f"""# Search Result: {result['title']}

**URL:** {result['url']}
**Query:** {query}
**Date:** {get_today_str()}

## Summary
{result['summary']}

## Raw Content
{result['raw_content'] if result['raw_content'] else 'No raw content available'}
"""
        
        files[filename] = file_content
        saved_files.append(filename)
        summaries.append(f"- {filename}: {result['summary']}...")
    
    # Create minimal summary for tool message - focus on what was collected
    summary_text = f"""🔍 Found {len(processed_results)} result(s) for '{query}':

{chr(10).join(summaries)}

Files: {', '.join(saved_files)}
💡 Use read_file() to access full details when needed."""

    return Command(
        update={
            "files": files,
            "messages": [
                ToolMessage(summary_text, tool_call_id=tool_call_id)
            ],
        }
    )

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:502: UserWarning: typing.NotRequired is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:502: UserWarning: typing.NotRequired is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


In [4]:

@tool(parse_docstring=True)
def think_tool(reflection: str) -> str:
    """Tool for strategic reflection on research progress and decision-making.

    Use this tool after each search to analyze results and plan next steps systematically.
    This creates a deliberate pause in the research workflow for quality decision-making.

    When to use:
    - After receiving search results: What key information did I find?
    - Before deciding next steps: Do I have enough to answer comprehensively?
    - When assessing research gaps: What specific information am I still missing?
    - Before concluding research: Can I provide a complete answer now?
    - How complex is the question: Have I reached the number of search limits?

    Reflection should address:
    1. Analysis of current findings - What concrete information have I gathered?
    2. Gap assessment - What crucial information is still missing?
    3. Quality evaluation - Do I have sufficient evidence/examples for a good answer?
    4. Strategic decision - Should I continue searching or provide my answer?

    Args:
        reflection: Your detailed reflection on research progress, findings, gaps, and next steps

    Returns:
        Confirmation that reflection was recorded for decision-making
    """
    return f"Reflection recorded: {reflection}"

## Deep Agent

Now, we can just apply all of our prior learnings: 

* We'll give the researcher a `think_tool` and our `search_tool` above.
* We'll give our parent agent `file tools`, a `think_tool`, and a `task` tool. 

In [ ]:
from datetime import datetime

from IPython.display import Image, display
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from utils import show_prompt, stream_agent

from tools_and_prompts.file_tools import ls, read_file, write_file
from tools_and_prompts.prompts import (
    FILE_USAGE_INSTRUCTIONS,
    RESEARCHER_INSTRUCTIONS,
    SUBAGENT_USAGE_INSTRUCTIONS,
    TODO_USAGE_INSTRUCTIONS,
)
from tools_and_prompts.state import DeepAgentState
from tools_and_prompts.task_tool import _create_task_tool

from tools_and_prompts.todo_tools import write_todos, read_todos

In [6]:
# Create agent using create_react_agent directly
model = init_chat_model(model="anthropic:claude-sonnet-4-5-20250929",temperature=0.5)

# Limits
max_concurrent_research_units = 3
max_researcher_iterations = 3

# Tools
sub_agent_tools = [tavily_search, think_tool]
built_in_tools = [ls, read_file, write_file, write_todos, read_todos, think_tool]

# Create research sub-agent
research_sub_agent = {
    "name": "research-agent",
    "description": "Delegate research to the sub-agent researcher. Only give this researcher one topic at a time.",
    "prompt": RESEARCHER_INSTRUCTIONS.format(date=get_today_str()),
    "tools": ["tavily_search", "think_tool"],
}

# Create task tool to delegate tasks to sub-agents
task_tool = _create_task_tool(
    sub_agent_tools, [research_sub_agent], model, DeepAgentState
)

delegation_tools = [task_tool]
all_tools = sub_agent_tools + built_in_tools + delegation_tools  # search available to main agent for trivial cases

# Build prompt
SUBAGENT_INSTRUCTIONS = SUBAGENT_USAGE_INSTRUCTIONS.format(
    max_concurrent_research_units=max_concurrent_research_units,
    max_researcher_iterations=max_researcher_iterations,
    date=datetime.now().strftime("%a %b %-d, %Y"),
)

INSTRUCTIONS = (
    "# TODO MANAGEMENT\n"
    + TODO_USAGE_INSTRUCTIONS
    + "\n\n"
    + "=" * 80
    + "\n\n"
    + "# FILE SYSTEM USAGE\n"
    + FILE_USAGE_INSTRUCTIONS
    + "\n\n"
    + "=" * 80
    + "\n\n"
    + "# SUB-AGENT DELEGATION\n"
    + SUBAGENT_INSTRUCTIONS
)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:502: UserWarning: typing.NotRequired is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:502: UserWarning: typing.NotRequired is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


In [7]:
from utils import format_messages
agent = create_react_agent(
    model, all_tools, prompt=INSTRUCTIONS, state_schema=DeepAgentState
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "How much should I feed my domastic short hair neutered cat?",
            }
        ],
    }
)

format_messages(result["messages"])

╭─────────────────────────────────────────────────── 🧑 Human ────────────────────────────────────────────────────╮
│ How much should I feed my domastic short hair neutered cat?                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│ I'll help you find information about feeding guidelines for a neutered domestic short hair cat. Let me start by │
│ organizing this research task.                                                                                  │
│                                                                                                                 │
│ 🔧 Tool Call: write_file                                                                                        │
│    Args: {                                                                                                      │
│   "file_path": "user_request.md",                                                                               │
│   "content": "# User Request\n\nHow much should I feed my domestic short hair neutered cat?\n\n## Key           │
│ Information Needed:\n- Feeding guidelines for domestic short hair cats\n- Specific considerations for neutered  │
│ cats\n- Portion sizes and frequency\n- Factors affecting feeding amounts (age, weight, activity level)\n"       │
│ }                                                                                                               │
│    ID: toolu_01VAaizxD2cy7QWYtQpLi9gf                                                                           │
│                                                                                                                 │
│ 🔧 Tool Call: write_todos                                                                                       │
│    Args: {                                                                                                      │
│   "todos": [                                                                                                    │
│     {                                                                                                           │
│       "content": "Research feeding guidelines for neutered domestic short hair cats",                           │
│       "status": "in_progress"                                                                                   │
│     },                                                                                                          │
│     {                                                                                                           │
│       "content": "Compile findings and provide comprehensive feeding recommendations",                          │
│       "status": "pending"                                                                                       │
│     }                                                                                                           │
│   ]                                                                                                             │
│ }                                                                                                               │
│    ID: toolu_015R2RmKEkCYwuYJ3QGEGQC9                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ Updated file user_request.md                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ Updated todo list to [{'content': 'Research feeding guidelines for neutered domestic short hair cats',          │
│ 'status': 'in_progress'}, {'content': 'Compile findings and provide comprehensive feeding recommendations',     │
│ 'status': 'pending'}]                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│ Now let me delegate the research to a specialized sub-agent:                                                    │
│                                                                                                                 │
│ 🔧 Tool Call: task                                                                                              │
│    Args: {                                                                                                      │
│   "description": "Research comprehensive feeding guidelines for neutered domestic short hair cats. Include:     │
│ daily calorie requirements, portion sizes based on weight, feeding frequency recommendations, differences       │
│ between wet and dry food, special considerations for neutered cats (metabolism changes, weight management), how │
│ age and activity level affect feeding amounts, and general veterinary recommendations.",                        │
│   "subagent_type": "research-agent"                                                                             │
│ }                                                                                                               │
│    ID: toolu_01CCF4bX5XMPmcSxa2Z4BbZX                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 🔧 Tool Call: read_todos                                                                                        │
│    Args: {}                                                                                                     │
│    ID: toolu_01Wd9S5Rx2h4Z3xXkGnqs45J                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ Current TODO List:                                                                                              │
│ 1. 🔄 Research feeding guidelines for neutered domestic short hair cats (in_progress)                           │
│ 2. ⏳ Compile findings and provide comprehensive feeding recommendations (pending)                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 🔧 Tool Call: write_todos                                                                                       │
│    Args: {                                                                                                      │
│   "todos": [                                                                                                    │
│     {                                                                                                           │
│       "content": "Research feeding guidelines for neutered domestic short hair cats",                           │
│       "status": "completed"                                                                                     │
│     },                                                                                                          │
│     {                                                                                                           │
│       "content": "Compile findings and provide comprehensive feeding recommendations",                          │
│       "status": "in_progress"                                                                                   │
│     }                                                                                                           │
│   ]                                                                                                             │
│ }                                                                                                               │
│    ID: toolu_01FyP6SWMpCgqK5uzJQGuXvL                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ Updated todo list to [{'content': 'Research feeding guidelines for neutered domestic short hair cats',          │
│ 'status': 'completed'}, {'content': 'Compile findings and provide comprehensive feeding recommendations',       │
│ 'status': 'in_progress'}]                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│ ## Feeding Your Neutered Domestic Short Hair Cat                                                                │
│                                                                                                                 │
│ Here's a comprehensive guide to help you feed your cat properly:                                                │
│                                                                                                                 │
│ ### **Quick Answer**                                                                                            │
│ For a neutered domestic short hair cat, feed **20-25 calories per pound of body weight per day**, divided into  │
│ 2-3 meals. This typically translates to:                                                                        │
│ - **Dry food**: ½ to 1 cup per day (depending on weight)                                                        │
│ - **Wet food**: 2-4 cans (3 oz each) per day                                                                    │
│ - **Combination**: Mix of both for optimal health                                                               │
│                                                                                                                 │
│ ### **Why Neutered Cats Need Special Attention**                                                                │
│ After neutering, your cat's metabolism decreases by 20-30%, but appetite often increases. This creates a higher │
│ risk of weight gain, so portion control is critical.                                                            │
│                                                                                                                 │
│ ### **Feeding Guidelines by Weight**                                                                            │
│                                                                                                                 │
│ | Cat Weight | Daily Calories | Dry Food | Wet Food |                                                           │
│ |------------|---------------|----------|----------|                                                            │
│ | 8 lbs | 160-200 cal | ½-⅔ cup | 2-3 cans |                                                                    │
│ | 10 lbs | 200-250 cal | ⅔-¾ cup | 2½-3½ cans |                                                                 │
│ | 12 lbs | 240-300 cal | ¾-1 cup | 3-4 cans |                                                                   │
│ | 15 lbs | 300-375 cal | 1-1¼ cups | 3½-5 cans |                                                                │
│                                                                                                                 │
│ ### **Best Practices**                                                                                          │
│                                                                                                                 │
│ 1. **Feed 2-3 times daily** - Avoid free feeding to prevent overeating                                          │
│ 2. **Measure portions precisely** - Use measuring cups or a kitchen scale                                       │
│ 3. **Choose quality food** - Look for "neutered cat" or "weight control" formulas                               │
│ 4. **Monitor weight monthly** - Adjust portions if your cat gains or loses weight                               │
│ 5. **Provide fresh water** - Especially important with dry food                                                 │
│                                                                                                                 │
│ ### **Wet vs. Dry Food**                                                                                        │
│ - **Wet food**: Better for hydration and weight managem